## 1. Cài đặt thư viện

In [ ]:
!pip install -q rouge-score bert-score sacrebleu bitsandbytes peft

## 2. Imports & Cấu hình

In [ ]:
import os, math
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ── Paths (Kaggle) ────────────────────────────────────────────────────────────
MODEL_NAME       = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   
LORA_ADAPTER_DIR = "/kaggle/input/eyelllama-lora"          
DATA_PATH        = "/kaggle/input/eye-disease-train-v2/eye_disease_train_v2.jsonl"
OUTPUT_DIR       = "/kaggle/working"

# ── Hyper-params ──────────────────────────────────────────────────────────────
RANDOM_SEED    = 42
TEST_RATIO     = 0.2
MAX_NEW_TOKENS = 256

# ── System prompt (giống project/model.py) ────────────────────────────────────
SYSTEM_PROMPT = (
    "You are EyeLlama, an AI assistant specialized in eye diseases and ophthalmology. "
    "You are not a human doctor and you are not any specific person. "
    "When asked who you are, always identify yourself as Eyesight. "
    "Provide helpful, accurate information about eye conditions, symptoms, diagnoses, and treatments."
)

matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 120})
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device  : {device}")
print(f"PyTorch : {torch.__version__}")
print(f"LoRA dir exists: {os.path.isdir(LORA_ADAPTER_DIR)}")
print(f"Data exists    : {os.path.isfile(DATA_PATH)}")

## 3. Load & Tách tập dữ liệu

In [ ]:
full_dataset = load_dataset("json", data_files=DATA_PATH)["train"]
split        = full_dataset.train_test_split(test_size=TEST_RATIO, seed=RANDOM_SEED)
test_dataset = split["test"]
test_samples = [test_dataset[i] for i in range(len(test_dataset))]
references   = [s["output"] for s in test_samples]

print(f"Tổng số mẫu : {len(full_dataset)}")
print(f"Tập test    : {len(test_samples)} mẫu")
print()
s0 = test_samples[0]
print(f"Ví dụ mẫu 0:")
print(f"  instruction: {s0['instruction']}")
print(f"  input      : {s0['input']}")
print(f"  output     : {s0['output'][:100]}...")

## 4. Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded:", MODEL_NAME)

## 5. Hàm sinh câu trả lời & tính Perplexity

In [ ]:
def build_prompt(sample: dict) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"{sample['instruction']}\n{sample['input']}"}
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_response(model, sample: dict, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """Greedy decoding — deterministic & reproducible."""
    prompt = build_prompt(sample)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def compute_perplexity(model, samples: list, batch_size: int = 4) -> float:
    """Perplexity trung bình trên tập samples."""
    total_loss, total_tokens = 0.0, 0
    for i in tqdm(range(0, len(samples), batch_size), desc="Perplexity"):
        batch = samples[i:i + batch_size]
        texts = []
        for s in batch:
            msgs = [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": f"{s['instruction']}\n{s['input']}"},
                {"role": "assistant", "content": s["output"]}
            ]
            texts.append(tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=False
            ))
        enc = tokenizer(
            texts, return_tensors="pt",
            truncation=True, max_length=512, padding=True
        ).to(model.device)
        labels = enc["input_ids"].clone()
        labels[labels == tokenizer.pad_token_id] = -100
        with torch.no_grad():
            loss = model(**enc, labels=labels).loss
        n_tok = (labels != -100).sum().item()
        total_loss   += loss.item() * n_tok
        total_tokens += n_tok
    return math.exp(total_loss / total_tokens)


print("Functions defined.")

## 6. Base Model — Sinh output & Perplexity

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("[1/2] Loading Base TinyLlama from HuggingFace...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = True
base_model.eval()
print("Base model ready.")

In [ ]:
print(f"Generating {len(test_samples)} responses (Base)...")
base_outputs = [
    generate_response(base_model, s)
    for s in tqdm(test_samples, desc="Base TinyLlama")
]
print("Sample:", base_outputs[0][:150])

In [ ]:
base_ppl = compute_perplexity(base_model, test_samples)
print(f"Base Perplexity: {base_ppl:.4f}")

In [ ]:
del base_model
torch.cuda.empty_cache()
print("Base model freed from GPU.")

## 7. EyeLlama (Fine-tuned) — Sinh output & Perplexity

In [ ]:
print("[2/2] Loading EyeLlama (Base + LoRA)...")
_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
    ),
    device_map="auto",
)
_base.config.use_cache = True

finetuned_model = PeftModel.from_pretrained(_base, LORA_ADAPTER_DIR)
finetuned_model.eval()
print("EyeLlama ready.")

In [ ]:
print(f"Generating {len(test_samples)} responses (EyeLlama)...")
finetuned_outputs = [
    generate_response(finetuned_model, s)
    for s in tqdm(test_samples, desc="EyeLlama")
]
print("Sample:", finetuned_outputs[0][:150])

In [ ]:
finetuned_ppl = compute_perplexity(finetuned_model, test_samples)
print(f"EyeLlama Perplexity: {finetuned_ppl:.4f}")

In [ ]:
df_results = pd.DataFrame({
    "instruction"      : [s["instruction"] for s in test_samples],
    "input"            : [s["input"]       for s in test_samples],
    "reference"        : references,
    "base_output"      : base_outputs,
    "finetuned_output" : finetuned_outputs,
})
csv_path = os.path.join(OUTPUT_DIR, "eval_results_full.csv")
df_results.to_csv(csv_path, index=False)
print(f"Saved {len(df_results)} rows → {csv_path}")
df_results.head(2)

## 8. BLEU Score

In [ ]:
import sacrebleu

def compute_bleu(predictions, references):
    return {
        f"BLEU-{n}": round(
            sacrebleu.corpus_bleu(predictions, [references],
                                  max_ngram_order=n, smooth_method="exp").score, 4
        )
        for n in [1, 2, 4]
    }

bleu_base      = compute_bleu(base_outputs,      references)
bleu_finetuned = compute_bleu(finetuned_outputs, references)

print(f"{'Metric':<10} {'Base':>10} {'EyeLlama':>12}")
print("-" * 35)
for k in bleu_base:
    print(f"{k:<10} {bleu_base[k]:>10.4f} {bleu_finetuned[k]:>12.4f}")

## 9. ROUGE Scores

In [ ]:
from rouge_score import rouge_scorer

def compute_rouge(predictions, references):
    sc = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)
    scores = {"rouge1":[], "rouge2":[], "rougeL":[]}
    for p, r in zip(predictions, references):
        res = sc.score(r, p)
        for k in scores:
            scores[k].append(res[k].fmeasure)
    return {k: round(float(np.mean(v)), 4) for k, v in scores.items()}

def rouge_per_sample(predictions, references):
    sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    return [sc.score(r, p)["rougeL"].fmeasure for p, r in zip(predictions, references)]

rouge_base      = compute_rouge(base_outputs,      references)
rouge_finetuned = compute_rouge(finetuned_outputs, references)
rougeL_base_list      = rouge_per_sample(base_outputs,      references)
rougeL_finetuned_list = rouge_per_sample(finetuned_outputs, references)

print(f"{'Metric':<12} {'Base':>10} {'EyeLlama':>12}")
print("-" * 37)
for k in rouge_base:
    label = k.upper().replace("ROUGE", "ROUGE-")
    print(f"{label:<12} {rouge_base[k]:>10.4f} {rouge_finetuned[k]:>12.4f}")

## 10. BERTScore

In [ ]:
from bert_score import score as bert_score_fn

def compute_bertscore(predictions, references, lang="en"):
    P, R, F1 = bert_score_fn(predictions, references, lang=lang,
                              verbose=False, batch_size=8)
    return {
        "BERTScore-P" : round(P.mean().item(),  4),
        "BERTScore-R" : round(R.mean().item(),  4),
        "BERTScore-F1": round(F1.mean().item(), 4),
    }

print("Computing BERTScore — Base...")
bert_base      = compute_bertscore(base_outputs,      references)
print("Computing BERTScore — EyeLlama...")
bert_finetuned = compute_bertscore(finetuned_outputs, references)

print(f"\n{'Metric':<16} {'Base':>10} {'EyeLlama':>12}")
print("-" * 41)
for k in bert_base:
    print(f"{k:<16} {bert_base[k]:>10.4f} {bert_finetuned[k]:>12.4f}")

## 11. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
w = 0.35

# BLEU
ax = axes[0]
bleu_lbl = ["BLEU-1", "BLEU-2", "BLEU-4"]
x = np.arange(len(bleu_lbl))
b1 = ax.bar(x-w/2, [bleu_base[k]      for k in bleu_lbl], w, label="Base",     color="#5b9bd5", alpha=0.85)
b2 = ax.bar(x+w/2, [bleu_finetuned[k] for k in bleu_lbl], w, label="EyeLlama", color="#ed7d31", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(bleu_lbl)
ax.set_ylabel("Score"); ax.set_title("BLEU Score", fontweight="bold"); ax.legend()
for bar in [*b1,*b2]: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

# ROUGE
ax2 = axes[1]
rouge_lbl = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
rv_base = [rouge_base["rouge1"], rouge_base["rouge2"], rouge_base["rougeL"]]
rv_ft   = [rouge_finetuned["rouge1"], rouge_finetuned["rouge2"], rouge_finetuned["rougeL"]]
x2 = np.arange(len(rouge_lbl))
b3 = ax2.bar(x2-w/2, rv_base, w, label="Base",     color="#5b9bd5", alpha=0.85)
b4 = ax2.bar(x2+w/2, rv_ft,   w, label="EyeLlama", color="#ed7d31", alpha=0.85)
ax2.set_xticks(x2); ax2.set_xticklabels(rouge_lbl)
ax2.set_ylabel("F1 Score"); ax2.set_title("ROUGE Score (F1)", fontweight="bold"); ax2.legend()
for bar in [*b3,*b4]: ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig_bleu_rouge.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BERTScore
ax = axes[0]
bert_lbl = ["Precision", "Recall", "F1"]
bv_base = [bert_base["BERTScore-P"], bert_base["BERTScore-R"], bert_base["BERTScore-F1"]]
bv_ft   = [bert_finetuned["BERTScore-P"], bert_finetuned["BERTScore-R"], bert_finetuned["BERTScore-F1"]]
x = np.arange(len(bert_lbl))
b1 = ax.bar(x-w/2, bv_base, w, label="Base",     color="#5b9bd5", alpha=0.85)
b2 = ax.bar(x+w/2, bv_ft,   w, label="EyeLlama", color="#ed7d31", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(bert_lbl)
ax.set_ylim(0.8, 1.0); ax.set_ylabel("Score")
ax.set_title("BERTScore", fontweight="bold"); ax.legend()
for bar in [*b1,*b2]: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001, f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)

# Perplexity
ax2 = axes[1]
ppl_lbl = ["TinyLlama\n(Base)", "EyeLlama\n(Fine-tuned)"]
ppls    = [base_ppl, finetuned_ppl]
bars = ax2.bar(ppl_lbl, ppls, color=["#5b9bd5","#ed7d31"], alpha=0.85, width=0.4)
for bar, val in zip(bars, ppls):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f"{val:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=11)
ax2.set_ylabel("Perplexity (thấp hơn = tốt hơn)")
ax2.set_title("Perplexity trên tập Test", fontweight="bold")
ax2.set_ylim(0, max(ppls)*1.25)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig_bertscore_ppl.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
df_box = pd.DataFrame({
    "TinyLlama (Base)"     : rougeL_base_list,
    "EyeLlama (Fine-tuned)": rougeL_finetuned_list,
})
df_box.boxplot(ax=ax, patch_artist=True,
               boxprops=dict(facecolor="lightblue"),
               medianprops=dict(color="red", linewidth=2))
ax.set_ylabel("ROUGE-L F1")
ax.set_title("Phân phối ROUGE-L theo từng mẫu test", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig_rougeL_dist.png"), dpi=150, bbox_inches="tight")
plt.show()

## 12. Bảng tổng hợp & Kết luận

In [ ]:
def pct(bv, fv, lower_better=False):
    d = (fv - bv) / abs(bv) * 100 if bv else 0
    d = -d if lower_better else d
    return f"{'+' if d>=0 else ''}{d:.2f}%"

rows = [
    ("BLEU-1",       bleu_base["BLEU-1"],       bleu_finetuned["BLEU-1"],       False),
    ("BLEU-2",       bleu_base["BLEU-2"],       bleu_finetuned["BLEU-2"],       False),
    ("BLEU-4",       bleu_base["BLEU-4"],       bleu_finetuned["BLEU-4"],       False),
    ("ROUGE-1 (F1)", rouge_base["rouge1"],      rouge_finetuned["rouge1"],      False),
    ("ROUGE-2 (F1)", rouge_base["rouge2"],      rouge_finetuned["rouge2"],      False),
    ("ROUGE-L (F1)", rouge_base["rougeL"],      rouge_finetuned["rougeL"],      False),
    ("BERTScore-F1", bert_base["BERTScore-F1"], bert_finetuned["BERTScore-F1"], False),
    ("Perplexity",   base_ppl,                  finetuned_ppl,                  True),
]

df_summary = pd.DataFrame(rows, columns=["Metric","TinyLlama (Base)","EyeLlama (Fine-tuned)","lower_better"])
df_summary["Cải thiện"] = [
    pct(r["TinyLlama (Base)"], r["EyeLlama (Fine-tuned)"], r["lower_better"])
    for _, r in df_summary.iterrows()
]
df_summary = df_summary.drop(columns=["lower_better"])
df_summary.to_csv(os.path.join(OUTPUT_DIR, "eval_summary.csv"), index=False)

print("═" * 65)
print("        KẾT QUẢ THỰC NGHIỆM — SO SÁNH METRICS")
print("═" * 65)
print(df_summary.to_string(index=False))
print("═" * 65)
print("Perplexity: thấp hơn = tốt hơn; các metric còn lại: cao hơn = tốt hơn")
df_summary

## 13. Ví dụ định tính

In [ ]:
NUM_EXAMPLES = 5
print("=" * 80)
for i in range(NUM_EXAMPLES):
    s = test_samples[i]
    print(f"\n[Mẫu {i+1}]")
    print(f"  Input    : {s['instruction']} | {s['input']}")
    print(f"  Reference: {s['output']}")
    print(f"  Base     : {base_outputs[i]}")
    print(f"  EyeLlama : {finetuned_outputs[i]}")
    print("-" * 80)

df_qual = df_results.iloc[:NUM_EXAMPLES][["instruction","input","reference","base_output","finetuned_output"]].copy()
df_qual.columns = ["Instruction","Input","Reference","TinyLlama Base","EyeLlama"]
df_qual.index   = [f"Mẫu {i+1}" for i in range(NUM_EXAMPLES)]
df_qual